# Smoke test — EXP-01: Mean pooling + linear

Kiem thu toan bo luong: doc du lieu -> huan luyen -> danh gia -> luu/tai checkpoint

- 500 mau train, 200 mau val
- 1 epoch
- Khong dung W&B
- Chay tren Colab T4 hoac CPU

## Cell 1 — Mount Drive va setup

In [ ]:
import os, sys

# --- Colab: bo comment dong duoi de mount Drive ---
# from google.colab import drive
# drive.mount('/content/drive')

# Clone repo neu chua co (chi chay lan dau)
# !git clone https://github.com/<username>/blip2-fusion-experiment-vqa.git /content/repo

# Duong dan goc repo — sua cho phu hop moi truong
REPO_ROOT = "/content/repo"          # Colab
# REPO_ROOT = r"d:\Du-an\blip2-fusion-experiment-vqa"  # Local Windows

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print("Working dir:", os.getcwd())

## Cell 2 — Cai dependencies

In [ ]:
!pip install -r requirements.txt -q

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Cell 3 — Cau hinh duong dan du lieu

In [ ]:
# ================================================================
# SUA CAC DUONG DAN NAY CHO PHU HOP MOI TRUONG CUA BAN
# ================================================================

DATA_ROOT    = "/content/drive/MyDrive/blip2_project/data"
VQAV2_DIR    = "vqav2"      # {DATA_ROOT}/vqav2/ chua file JSON
CACHE_DIR    = "cache"      # {DATA_ROOT}/cache/ chua *.h5
ANSWER_LIST  = f"{DATA_ROOT}/ans2idx.json"
OUTPUT_DIR   = "/content/checkpoints/exp01_smoke"

# Kich thuoc smoke test
TRAIN_SIZE   = 500
VAL_SIZE     = 200
BATCH_SIZE   = 32
NUM_EPOCHS   = 1

# ================================================================
print("DATA_ROOT   :", DATA_ROOT)
print("ANSWER_LIST :", ANSWER_LIST)
print("OUTPUT_DIR  :", OUTPUT_DIR)
print("Train size  :", TRAIN_SIZE, "| Val size:", VAL_SIZE)

## Cell 4 — Pre-extract CLIP features (bo qua neu da co cache)

In [ ]:
import os
train_h5 = os.path.join(DATA_ROOT, CACHE_DIR, "train_features.h5")
val_h5   = os.path.join(DATA_ROOT, CACHE_DIR, "val_features.h5")

if os.path.exists(train_h5) and os.path.exists(val_h5):
    print("Cache da ton tai — bo qua pre-extract.")
    import h5py
    with h5py.File(train_h5, "r") as f:
        print(f"  train_features.h5: {len(f.keys()):,} anh")
    with h5py.File(val_h5, "r") as f:
        print(f"  val_features.h5  : {len(f.keys()):,} anh")
else:
    print("Chua co cache — chay pre-extract (chi 500 anh de smoke test)...")
    !python data/pre_extract_features.py \
        --split both \
        --data_root "{DATA_ROOT}" \
        --output_dir "{DATA_ROOT}/{CACHE_DIR}" \
        --vqav2_dir "{VQAV2_DIR}" \
        --max_images 500 \
        --batch_size 64
    print("Pre-extract hoan thanh.")

## Cell 5 — Xay dung config + DataLoader

In [ ]:
from omegaconf import OmegaConf
from data.vqa_dataset import build_dataloader

cfg_dict = {
    "model": {
        "name": "mean_linear",
        "vision_width": 1024,
        "text_dim": 768,
        "hidden_size": 768,
        "num_answers": 3129,
        "fusion_output_size": 1024,
        "dropout": 0.1,
        "num_layers": 12,
        "num_heads": 12,
        "intermediate_size": 3072,
        "num_query_tokens": 32,
    },
    "data": {
        "data_root":           DATA_ROOT,
        "vqav2_dir":           VQAV2_DIR,
        "coco_dir":            "coco",
        "cache_dir":           CACHE_DIR,
        "answer_list":         ANSWER_LIST,
        "train_size":          TRAIN_SIZE,
        "val_size":            VAL_SIZE,
        "batch_size":          BATCH_SIZE,
        "max_question_length": 50,
        "image_size":          224,
        "num_workers":         2,
        "seed":                42,
    },
    "training": {
        "output_dir":                  OUTPUT_DIR,
        "log_dir":                     OUTPUT_DIR + "/logs",
        "num_epochs":                  NUM_EPOCHS,
        "eval_batch_size":             BATCH_SIZE,
        "learning_rate":               1e-4,
        "weight_decay":                0.05,
        "warmup_steps":                10,
        "gradient_clip":               1.0,
        "gradient_accumulation_steps": 1,
        "save_every":                  1,
        "eval_every":                  1,
        "seed":                        42,
        "mixed_precision":             True,
        "resume_from":                 None,
    },
    "optimizer": {"name": "adamw", "betas": [0.9, 0.999], "eps": 1e-8},
    "scheduler": {"name": "cosine", "min_lr": 1e-6},
    "logging":   {"use_wandb": False, "project": "blip2-vqa", "run_name": "exp01_smoke"},
}

config = OmegaConf.create(cfg_dict)

train_loader = build_dataloader("train", config, use_cache=True)
val_loader   = build_dataloader("val",   config, use_cache=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# Kiem tra 1 batch
batch = next(iter(train_loader))
print("\nBatch keys:", list(batch.keys()))
print("image_features shape:", batch["image_features"].shape)   # [B, 257, 1024]
print("input_ids shape      :", batch["input_ids"].shape)        # [B, 50]
print("answer_scores shape  :", batch["answer_scores"].shape)    # [B, 3129]
print("answer_type sample   :", batch["answer_type"][:3])

## Cell 6 — Huan luyen 1 epoch va kiem tra loss giam

In [ ]:
import torch
from models import build_model, FrozenTextEncoder
from training.losses import VQALoss
from training.trainer import VQATrainer
from utils.helpers import build_optimizer, build_scheduler, set_seed

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Khoi tao model EXP-01
model        = build_model(config)
text_encoder = FrozenTextEncoder()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")

# Optimizer & scheduler
optimizer  = build_optimizer(model, cfg_dict)
num_steps  = len(train_loader) * NUM_EPOCHS
scheduler  = build_scheduler(optimizer, cfg_dict, num_steps)

# Trainer
trainer = VQATrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=VQALoss(loss_type="bce"),
    device=device,
    output_dir=OUTPUT_DIR,
    gradient_accumulation_steps=1,
    gradient_clip=1.0,
    mixed_precision=torch.cuda.is_available(),
    text_encoder=text_encoder,
    log_every=5,
)

print("\nBat dau huan luyen...")
trainer.train(num_epochs=NUM_EPOCHS, start_epoch=0)
print("\nHuan luyen hoan thanh.")

## Cell 7 — Kiem tra so lieu (loss, accuracy) in ra man hinh

In [ ]:
# Chay danh gia thu cong de xem ket qua ro rang
from configs.contracts import (
    KEY_LOSS, KEY_OVERALL_ACC, KEY_YESNO_ACC, KEY_NUMBER_ACC, KEY_OTHER_ACC
)

print("Dang chay danh gia tren val set...")
results = trainer.evaluate()

print("\n" + "="*50)
print("KET QUA SMOKE TEST — EXP-01")
print("="*50)
print(f"Val Loss          : {results[KEY_LOSS]:.4f}")
print(f"Val Accuracy (Overall) : {results.get(KEY_OVERALL_ACC, 0)*100:.2f}%")
print(f"Val Accuracy (Yes/No)  : {results.get(KEY_YESNO_ACC,  0)*100:.2f}%")
print(f"Val Accuracy (Number)  : {results.get(KEY_NUMBER_ACC, 0)*100:.2f}%")
print(f"Val Accuracy (Other)   : {results.get(KEY_OTHER_ACC,  0)*100:.2f}%")
print("="*50)
print("\nGhi vao bang theo doi (exp01_lan1):")
print(f"  Run Name   : exp01_lan1")
print(f"  Train Loss : (xem log o tren)")
print(f"  Val Loss   : {results[KEY_LOSS]:.4f}")
print(f"  Val Acc    : {results.get(KEY_OVERALL_ACC, 0)*100:.2f}%")
print(f"  Best Epoch : 1 / 1")

## Cell 8 — Luu va tai lai checkpoint (kiem tra resume)

In [ ]:
import os
from pathlib import Path

# Tim checkpoint da luu
ckpt_dir  = Path(OUTPUT_DIR)
ckpts     = sorted(ckpt_dir.glob("checkpoint_epoch_*.pth"))
best_path = ckpt_dir / "best_model.pth"

print("Checkpoint da luu:")
for p in ckpts:
    size_mb = p.stat().st_size / 1e6
    print(f"  {p.name}  ({size_mb:.2f} MB)")
if best_path.exists():
    print(f"  best_model.pth  ({best_path.stat().st_size/1e6:.2f} MB)")

# Tai lai checkpoint va kiem tra
if ckpts:
    ckpt_path = str(ckpts[-1])
    print(f"\nTai lai tu: {ckpt_path}")
    
    state = torch.load(ckpt_path, map_location=device)
    print("Keys trong checkpoint:", list(state.keys()))
    print("Epoch da luu         :", state["epoch"])
    print("Global step          :", state["global_step"])
    print("Best val metric      :", state["best_val_metric"])
    
    # Thu load vao trainer moi (gia lap resume)
    model2        = build_model(config)
    text_encoder2 = FrozenTextEncoder()
    optimizer2    = build_optimizer(model2, cfg_dict)
    scheduler2    = build_scheduler(optimizer2, cfg_dict, num_steps)
    
    trainer2 = VQATrainer(
        model=model2,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer2,
        scheduler=scheduler2,
        loss_fn=VQALoss(loss_type="bce"),
        device=device,
        output_dir=OUTPUT_DIR + "/resume_test",
        text_encoder=text_encoder2,
    )
    
    start_epoch = trainer2.load_checkpoint(ckpt_path)
    print(f"\nResume thanh cong tu epoch {start_epoch}")
    print("Global step sau load:", trainer2.global_step)
    print("Best val metric      :", trainer2.best_val_metric)
    
    print("\n" + "="*50)
    print("SMOKE TEST PASS - Tat ca buoc deu chay dung!")
    print("  [OK] Doc du lieu (DataLoader)")
    print("  [OK] Huan luyen 1 epoch (loss in ra man hinh)")
    print("  [OK] Danh gia (accuracy theo tung loai cau hoi)")
    print("  [OK] Luu checkpoint (.pth)")
    print("  [OK] Tai lai checkpoint (resume)")
    print("="*50)
else:
    print("CANH BAO: Khong tim thay checkpoint. Kiem tra OUTPUT_DIR.")